# 06 — Individual Risk Modeling (raw microdata)

The rate sources (notebooks 01–05) are **aggregate** — one number per group-year. This notebook uses the **raw NCHS record-level microdata** (one row per birth, ~12M births pooled over 2011–2013) to ask a different question:

> *What makes an individual birth more likely to end in infant death?*

That is the patient-level, risk-factor lens — birth weight, gestation, prenatal care, maternal age — rather than a time trend.

**Note:** this downloads ~500 MB on first run and pools millions of rows, so it is slower than the other notebooks. Deaths are double-counted by ~0.6% (they appear among the births too); negligible for modeling.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from src.data.sources import nber_microdata

In [ ]:
# downloads + pools 2011-2013 birth records (cached in data/raw/ after the first run)
df = nber_microdata.tidy()
print('births:', len(df), '| infant deaths:', int(df['died'].sum()))
print('overall IMR per 1,000:', round(df['died'].mean() * 1000, 2))
df.head()

In [ ]:
# infant mortality by birth-weight category - the dominant risk factor
bw = pd.cut(df['birthweight_g'], [0, 1500, 2500, 4000, 9000],
            labels=['<1500g', '1500-2499g', '2500-3999g', '4000g+'])
(df.groupby(bw, observed=True)['died'].mean() * 1000).round(1)

In [ ]:
# infant mortality by gestational age (preterm vs term)
gw = pd.cut(df['gestation_wks'], [0, 28, 32, 37, 50],
            labels=['<28wk', '28-31wk', '32-36wk', '37wk+'])
(df.groupby(gw, observed=True)['died'].mean() * 1000).round(1)

## Logistic regression: individual death risk

Model `died` on the birth-certificate covariates. Birth weight and gestation are expected to dominate.

In [ ]:
# drop rows missing a modeled covariate, then fit (sample for speed if needed)
cols = ['died', 'birthweight_g', 'gestation_wks', 'mother_age', 'prenatal_visits',
        'plurality', 'smoker']
model_df = df[cols].dropna()
fit = smf.logit('died ~ birthweight_g + gestation_wks + mother_age + prenatal_visits + '
                'plurality + smoker', data=model_df).fit(disp=0)
print(fit.summary())

In [ ]:
# odds ratios (easier to read than log-odds)
np.exp(fit.params).round(4)

## Notes

- **Birth weight and gestational age dominate** — risk rises steeply for low-birth-weight and very preterm infants, far more than any social factor here.
- This individual-level view complements the aggregate trends: the rates in notebooks 01–05 are the population average of exactly this risk surface.
- Birth weight and gestation are highly collinear (small, early babies), so their separate coefficients should be read together, not in isolation.
- Observational data: associations, not causal effects.